> 📅 __Date: 2026-08-21__

# 🤗 **BERT Fine-Tuning for Text Classification**

> **BERT Fine-Tuning = Taking a pretrained BERT model and adapting its parameters to a specific downstream task using task-specific data.**

This note walks through a complete **BERT text-classification workflow**:

```text
Raw Dataset
    ↓
Select Useful Columns
    ↓
Remove Null Values
    ↓
Map Original Classes
    ↓
Text Preprocessing
    ↓
Check Class Distribution
    ↓
Undersampling
    ↓
Convert to Hugging Face Dataset
    ↓
Train / Test Split
    ↓
Load BERT Tokenizer + Model
    ↓
Tokenization
    ↓
Training Configuration
    ↓
Evaluation Metrics
    ↓
Fine-Tune BERT
    ↓
Save Model
    ↓
Reload Model
    ↓
Make Predictions
```

---

# 🏋️ **Training vs Fine-Tuning**

Before understanding BERT fine-tuning, it is important to distinguish **training from scratch** and **fine-tuning**.

## **Training**

```text
-- Parameters are initialized randomly
-- Model learns parameters from training data
```

**When training from scratch:**

```text
Randomly initialized parameters
            ↓
Large amount of training data
            ↓
Optimization
            ↓
Learned model
```

The model has no pretrained language knowledge at the beginning.

> **Training from scratch generally requires much more data and computation because the model has to learn useful representations from the beginning.**

---

# 🔧 **Fine-Tuning**

```text
-- Parameters are initialized from a pre-trained model
-- Parameters are fine-tuned on new data
```

**The idea is:**

```text
Pretrained BERT
      ↓
Already learned language representations
      ↓
Task-specific dataset
      ↓
Fine-tuning
      ↓
BERT adapted to the new task
```

This is the main advantage of transfer learning:

> **Instead of learning language understanding from zero, we start from a model that has already learned useful language representations.**

For this notebook, the downstream task is:

> **Classifying consumer complaints into financial-product categories.**

---

# 🧠 **Why Fine-Tuning Works Well**

**A pretrained language model has already learned many reusable patterns:**

```text
Word relationships
Sentence structure
Contextual representations
Language patterns
```

Fine-tuning adjusts those learned representations so they become useful for a particular task.

**Conceptually:**

```text
General language knowledge
            ↓
       BERT
            ↓
Task-specific examples
            ↓
Fine-tuned BERT
            ↓
Task prediction
```

> **Fine-tuning does not mean the pretrained model is discarded. It means the pretrained parameters become the starting point for task-specific learning.**

---

# 📂 **Data Gathering**

**The dataset is loaded using pandas:**

```python
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/NLP and Gen-AI Batches/27 Jun 26 -  Velocity GenAI & Agentic AI Batch/Extra Notes/Dataset/complaints.csv"
)

df.head()
```

The dataset contains information about consumer complaints.

For this classification task, two columns are important:

```text
Product
Consumer complaint narrative
```

---

# 🧹 **Drop Other Columns**

```python
df = df[["Product", "Consumer complaint narrative"]]
```

This keeps only the columns required for the downstream task.

**Conceptually:**

```text
Original dataset
      ↓
Keep relevant columns
      ↓
Product + Complaint Narrative
```

**Here:**

```text
Product
→ Target / class information

Consumer complaint narrative
→ Text input given to BERT
```

---

# ❌ **Drop Null Values**

```python
df = df.dropna()
```

Null values are removed so that the training examples contain the information required by the preprocessing and tokenization pipeline.

**For this task, a useful training example should contain:**

```text
Complaint text
+
Target class
```

---

# 🏷️ **Class Mapping**

The original dataset contains many fine-grained product names.

For example:

```text
Vehicle loan or lease

Mortgage

Credit card

Student loan

Checking or savings account

...
```

Instead of keeping every original category as a separate class, the notebook groups related products into broader classes.

```python
class_dict = {

    'Vehicle loan or lease': 'Loan',

    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit_Reporting',

    'Credit card or prepaid card': 'Card',

    'Money transfer, virtual currency, or money service': 'Service',

    'Mortgage': 'Loan',

    'Payday loan, title loan, or personal loan': 'Loan',

    'Debt collection': "Others",

    'Checking or savings account': 'Others',

    'Student loan': 'Loan',

    'Consumer Loan': 'Loan',

    'Money transfers': 'Service',

    'Credit card': 'Card',

    'Bank account or service': 'Service',

    'Credit reporting': 'Credit_Reporting',

    'Prepaid card': 'Card',

    'Payday loan': 'Loan',

    'Other financial service': 'Service',

    'Virtual currency': 'Service'

}
```

**The mapping creates five broader classes:**

```text
Loan
Credit_Reporting
Card
Service
Others
```

---

# 🔄 **Apply the Class Mapping**

**The notebook uses:**

```python
# df["Product"].replace(class_dict, inplace=True)
df.replace({"Product": class_dict}, inplace=True)
```

The commented line shows an alternative form, while the second line performs the replacement.

**Conceptually:**

```text
Original Product
      ↓
class_dict
      ↓
Broader Product Category
```

**For example:**

```text
Mortgage
   ↓
Loan

Credit card
   ↓
Card

Credit reporting
   ↓
Credit_Reporting
```

---

# 🧽 **Text Preprocessing**

The complaint narratives are cleaned before being sent to the model.

```python
import re

def Preprocessing(complaint):

    ## Lower Casing :
    complaint = complaint.lower()

    ## Remove Masking :
    complaint = re.sub(r'[x]{2,}', '', complaint)

    ## Remove non-alpha :
    complaint = re.sub(r'[^a-z\s]', '', complaint)

    return complaint
```

**The function is then applied:**

```python
df['Consumer complaint narrative'] = (
    df['Consumer complaint narrative']
    .apply(Preprocessing)
)
```

---

# 🔡 **Preprocessing Step 1 — Lower Casing**

```python
complaint = complaint.lower()
```

**Example:**

```text
"I HAVE A PROBLEM"
        ↓
"i have a problem"
```

This reduces differences caused only by capitalization.

---

# 🕵️ **Preprocessing Step 2 — Remove Masking**

```python
complaint = re.sub(r'[x]{2,}', '', complaint)
```

This removes sequences of two or more `x` characters.

**For example:**

```text
xxxx
xxxxxx
xxxxxxxx
```

are removed.

**This is useful because datasets containing customer complaints can contain masked information such as:**

```text
XXXX
XXXXXX
```

The preprocessing removes these masking artifacts.

---

# ✂️ **Preprocessing Step 3 — Remove Non-Alphabetic Characters**

```python
complaint = re.sub(r'[^a-z\s]', '', complaint)
```

**This keeps:**

```text
a-z
spaces
```

and removes other characters.

**Conceptually:**

```text
Text
 ↓
Remove unwanted symbols / digits / punctuation
 ↓
Clean text
```

> **Important:** This preprocessing is specific to this notebook. BERT itself uses a tokenizer capable of handling punctuation and other token patterns, so aggressive preprocessing is a modeling choice rather than a universal BERT requirement.

---

# 📊 **Check the Class Distribution**

```python
df["Product"].value_counts()
```

This tells us how many examples are present in each class.

The notebook observes:

> **Data is imbalance**

More precisely:

> **The dataset is imbalanced when some classes contain many more examples than others.**

**Conceptually:**

```text
Loan              → many examples
Card              → fewer examples
Service           → ...
Others            → ...
Credit_Reporting  → ...
```

An imbalanced dataset can make a classifier favor majority classes.

---

# ⚖️ **Why Data Imbalance Matters**

**Suppose a dataset contains:**

```text
Class A → 9000 samples
Class B → 1000 samples
```

A model could obtain high overall accuracy by predicting Class A frequently, even if it performs poorly on Class B.

Therefore, class distribution should be inspected before training.

**Useful metrics for imbalanced classification include:**

```text
Precision
Recall
F1 Score
```

The notebook later uses **macro-averaged precision, recall, and F1**, which gives each class equal importance in the metric calculation.

---

# 💻 **BERT and Data Size**

**The notebook notes:**

> **BERT (or any pretrained model) will give better result on less data**

The important idea behind this statement is **transfer learning**:

```text
Pretrained BERT
→ already contains learned language representations
→ task-specific fine-tuning can work with less task-specific data
```

**However, the exact performance still depends on:**

```text
Dataset quality
Task difficulty
Class balance
Number of examples
Preprocessing
Hyperparameters
Evaluation setup
```

**The notebook also notes:**

> **For finetuning on more data computational resources required**

**This reflects an important practical trade-off:**

```text
More training data
      ↓
Potentially more useful training signal
      ↓
More computation / memory / time
```

---

# 📉 **Undersampling**

Because the dataset is imbalanced, the notebook creates an equally sized sample from each class.

```python
sample_df = pd.DataFrame()

for department in df['Product'].unique():

    temp_df = df[df['Product'] == department].sample(1000)

    sample_df = pd.concat(
        [sample_df, temp_df],
        axis=0
    )
```

**Then:**

```python
sample_df['Product'].value_counts()
```

---

# 🧮 **How Undersampling Works**

**For every class:**

```text
Select 1000 examples
```

Then concatenate the samples.

**Conceptually:**

```text
Loan
→ 1000

Credit_Reporting
→ 1000

Card
→ 1000

Service
→ 1000

Others
→ 1000
```

**Therefore the resulting dataset contains:**

$$
5 \times 1000 = 5000
$$

examples, assuming all five classes have at least 1000 examples available.

> **Undersampling reduces the majority classes rather than increasing the minority classes.**

### ✅ **Advantage**

```text
More balanced training data
```

### ⚠️ **Trade-off**

```text
Some original examples from majority classes are discarded
```

So undersampling can reduce training-data size in exchange for a more balanced class distribution.

---

# 🔎 **Check Unique Classes**

```python
df['Product'].unique()
```

This verifies which class names are present after the mapping step.

---

# 🔢 **Convert Class Names to Numeric Labels**

**The notebook creates:**

```python
classes = {

    "Loan": 0,

    "Credit_Reporting": 1,

    "Card": 2,

    "Service": 3,

    "Others": 4

}
```

**Then:**

```python
sample_df.replace({"Product": classes}, inplace=True)
```

Now the target values become:

```text
Loan              → 0
Credit_Reporting  → 1
Card              → 2
Service            → 3
Others             → 4
```

This converts human-readable class names into integer IDs that can be used by the classification model.

---

# 👀 **Inspect the Sampled Data**

```python
sample_df.head()
```

**At this point the dataset contains:**

```text
Product
Consumer complaint narrative
```

with the `Product` column represented using numeric class IDs.

---

# 📚 **Datasets**

## **Convert the Data to Hugging Face Dataset Format**

**The notebook installs the Hugging Face `datasets` library:**

```python
!pip install datasets
```

**Then:**

```python
from datasets import Dataset
```

---

# 🔄 **Pandas DataFrame → Hugging Face Dataset**

```python
dataset = Dataset.from_pandas(sample_df)

type(dataset)
```

**The notebook then displays:**

```python
dataset
```

The purpose of this conversion is to move from a pandas-centric workflow to the data representation expected by many Hugging Face training utilities.

**Conceptually:**

```text
pandas.DataFrame
        ↓
Dataset.from_pandas()
        ↓
Hugging Face Dataset
```

---

# 🏷️ **Target Column Should Be Named `label`**

**The notebook notes:**

> **Target column must be named as "label"**

**and:**

> **Other names such as "Product" is not accepted**

**In this workflow, the column is later renamed to the Hugging Face convention used by the `Trainer`:**

```python
tokenize_data = tokenize_data.rename_column(
    "Product",
    "labels"
)
```

**So the notebook ultimately uses:**

```text
labels
```

as the training target column.

**This distinction is worth remembering:**

```text
Notebook's original target column
→ Product

Renamed target column before Trainer
→ labels
```

---

# ✂️ **Train / Test Split**

```python
train_test = dataset.train_test_split(test_size=0.2)

train_test
```

The split reserves:

```text
80% → Training

20% → Test
```

For the 5000-example dataset:

$$
5000 \times 0.8 = 4000
$$

and:

$$
5000 \times 0.2 = 1000
$$

**The displayed output is:**

```text
DatasetDict({
    train: Dataset({
        features: ['Product', 'Consumer complaint narrative', '__index_level_0__'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['Product', 'Consumer complaint narrative', '__index_level_0__'],
        num_rows: 1000
    })
})
```

---

# 🧠 **Why Split the Dataset?**

The model should not be evaluated only on the same examples used for training.

```text
Training data
→ Used to learn

Test data
→ Used to evaluate generalization
```

**So:**

```text
Dataset
   ↓
Train + Test
   ↓
Train BERT on train
   ↓
Evaluate on unseen test examples
```

---

# 🤗 **Model & Tokenizer**

**The checkpoint is:**

```python
checkpoint = "bert-base-cased"
```

**Then:**

```python
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
```

---

# 🔤 **Load the Tokenizer**

```python
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
```

The tokenizer converts raw text into the token representation expected by BERT.

**Conceptually:**

```text
Raw complaint text
        ↓
Tokenizer
        ↓
Tokens
        ↓
Token IDs
        ↓
BERT
```

---

# 🧠 **Load BERT for Classification**

```python
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=5
)
```

**This loads:**

```text
bert-base-cased
```

**and attaches a sequence-classification head configured for:**

```text
5 classes
```

**The overall architecture becomes conceptually:**

```text
Complaint Text
      ↓
BERT Encoder
      ↓
Sequence Representation
      ↓
Classification Head
      ↓
5 Class Scores
      ↓
Predicted Class
```

---

# 🧩 **What Happens to the Classification Head?**

When using a pretrained checkpoint for a new classification task, the pretrained checkpoint may not contain a classification head matching the required number of labels.

**That is why a warning such as:**

```text
MISSING: those params were newly initialized because missing from the checkpoint.
Consider training on your downstream task.
```

can appear.

**The correct interpretation is:**

```text
BERT body
→ Loaded from pretrained checkpoint

New classification head
→ Newly initialized

Fine-tuning
→ Learns the new classification head + adapts model parameters
```

> **This warning is expected when the downstream task-specific head is not present in the original checkpoint.**

---

# 🔢 **Classifier Head Parameter Count**

BERT produces a representation that is passed to the **classification head**.

In this simplified illustration, suppose the representation has:

```text
8 neurons / representation values
```

## **Default Binary Classification**
```text
2 output classes
```

**A fully connected layer with:**
```text
Input neurons  = 8
Output classes = 2
```

**has:**
$$
8×2+2
$$

**where:**
```text
8 × 2 → Weight parameters
2     → Bias parameters
```

**Therefore:**
$$
8×2+2=18
$$

So the classifier head has **18 parameters** in this simplified example.

---

## **Our 5-Class Classification**

**Our complaint dataset has 5 classes:**

```text
Loan
Credit_Reporting
Card
Service
Others
```

**Therefore, we configure:**

```text
num_labels=5
```

**Now the classification head changes from:**
```text
8 → 2
```

**to:**
```text
8 → 5
```

**The parameter count becomes:**
$$
8×5+5
$$

**where:**
```text
8 × 5 → Weight parameters
5     → Bias parameters
```

**Therefore:**
$$
8×5+5=45
$$

So in this simplified example, the new 5-class classifier head has **45 parameters**.

---

## 🧠 **Visual Intuition**

```
             BERT Representation
                  8 neurons
              ┌─────────────┐
              │ 8 values    │
              └──────┬──────┘
                     │
             Classification Head
                     │
          ┌──────────┴──────────┐
          ↓                     ↓

      Default BERT          Our Task
      Binary Task           5 Classes

        8 → 2                 8 → 5
       outputs                outputs

     8 × 2 + 2              8 × 5 + 5
        = 18                   = 45
```
> **Key idea:** num_labels=5 tells the classification head that our downstream task has **5 possible output classes**, so the final layer must produce **5 class scores instead of 2**.

---

**For a linear classification layer, the general parameter-count formula is:**

$$
\text{Parameters}
=
(\text{input features} \times \text{output classes})
+
\text{output classes}
$$

The second term represents the bias parameters.

**For a classifier with input dimension \(H\) and \(C\) classes:**

$$
\text{Parameters}
=
H \times C + C
$$

**The notebook's:**

```text
8 × 5 + 5
```

follows that general linear-layer counting pattern for an **8-dimensional input** and **5 output classes**.

---

# 📏 **BERT Positional Encoding / Maximum Sequence Length**

**The notebook notes:**

> **BERT : Learned positional encoding (512)**

This means the BERT-base architecture uses learned position embeddings with a maximum sequence length of **512 positions** in the standard configuration.

Therefore, a text sequence longer than the configured maximum length needs to be handled using truncation or another strategy.

---

# 🔤 **Tokenization Function**

**The notebook defines:**

```python
def tokenize_function(row):

    return tokenizer(
        row["Consumer complaint narrative"],
        padding="max_length",
        max_length=500,
        truncation=True
    )
```

This performs three important operations.

---

# 📏 **`max_length=500`**

```python
max_length=500
```

The model receives sequences of up to 500 tokens in this notebook.

Because standard BERT supports up to 512 positions, 500 leaves room for the model's special-token handling while staying below the 512-position limit.

---

# ✂️ **`truncation=True`**

```python
truncation=True
```

**If a complaint is longer than the configured maximum length:**

```text
Long complaint
      ↓
Tokenizer
      ↓
Cut to maximum allowed length
```

The excess tokens are discarded.

This is necessary because the model cannot accept arbitrarily long sequences.

---

# 📦 **`padding="max_length"`**

```python
padding="max_length"
```

Shorter examples are padded until they reach the configured maximum sequence length.

**Conceptually:**

```text
Short text
   ↓
Tokens
   ↓
Padding tokens
   ↓
Exactly 500 positions
```

This creates fixed-size sequences for this notebook's batching setup.

---

# 🗂️ **Apply Tokenization to the Dataset**

```python
tokenize_data = train_test.map(
    tokenize_function,
    batched=True
)
```

**The resulting dataset is displayed as:**

```text
DatasetDict({
    train: Dataset({
        features: [
            'Product',
            'Consumer complaint narrative',
            '__index_level_0__',
            'input_ids',
            'token_type_ids',
            'attention_mask'
        ],
        num_rows: 4000
    })
    test: Dataset({
        features: [
            'Product',
            'Consumer complaint narrative',
            '__index_level_0__',
            'input_ids',
            'token_type_ids',
            'attention_mask'
        ],
        num_rows: 1000
    })
})
```

---

# 🧩 **What Are These Tokenizer Outputs?**

## **`input_ids`**

Token IDs representing the text.

```text
Text
 ↓
Tokenizer
 ↓
Integer token IDs
```

These IDs are passed into BERT.

---

## **`attention_mask`**

The attention mask identifies which positions correspond to actual input tokens versus padding.

**Conceptually:**

```text
1 → real token

0 → padding
```

**Example:**

```text
Token:   I   have   a   loan   [PAD]
Mask:    1    1     1    1      0
```

---

## **`token_type_ids`**

For BERT-style inputs, token type IDs can distinguish segments.

**Conceptually:**

```text
Sentence A → segment 0
Sentence B → segment 1
```

For a single complaint-text classification example, these often represent a single segment.

---

# ⚙️ **Hyperparameters for Training**

**The notebook uses Hugging Face `TrainingArguments`:**

```python
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",  ## Path to save trained model
    num_train_epochs=1,      ## How many epochs to train
)
```

---

# 📁 **`output_dir`**

```python
output_dir="./results"
```

This specifies the directory used by the trainer for training outputs.

**Conceptually:**

```text
Training
   ↓
Outputs / checkpoints / trainer artifacts
   ↓
./results
```

---

# 🔁 **`num_train_epochs`**

```python
num_train_epochs=1
```

One epoch means the training loop processes the training dataset once.

```text
Epoch 1
→ See training examples once
```

In practice, the number of epochs is a hyperparameter and should be selected based on validation performance, compute budget, and signs of underfitting or overfitting.

---

# 📊 **Evaluation Metrics**

**The notebook imports:**

```python
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
```

**Then defines:**

```python
def compute_metrics(pred):

    labels = pred.label_ids  ## Actual value

    preds = pred.predictions.argmax(-1) ## Predicted value

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="macro"
        )
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }
```

---

# 🎯 **Actual Labels**

```python
labels = pred.label_ids
```

These are the ground-truth class IDs.

**For example:**

```text
0
1
2
3
4
```

---

# 🤖 **Predicted Labels**

```python
preds = pred.predictions.argmax(-1)
```

The model produces scores/logits for each class.

**For example:**

```text
Class 0 → 1.2
Class 1 → 0.4
Class 2 → 3.7
Class 3 → 0.9
Class 4 → 0.2
```

**`argmax(-1)` selects the index of the largest score:**

```text
Largest score → Class 2
```

So:

> **`argmax` converts the model's class scores into the predicted class ID.**

---

# 📐 **Macro Precision, Recall, and F1**

**The notebook uses:**

```python
average="macro"
```

Macro averaging calculates the metric independently for each class and then averages the class-level results.

**Conceptually:**

$$
\text{Macro F1}
=
\frac{1}{C}
\sum_{c=1}^{C}
F1_c
$$

where \(C\) is the number of classes.

This is particularly useful when class balance matters because each class contributes equally to the final macro score.

---

# ✅ **Accuracy**

```python
acc = accuracy_score(labels, preds)
```

**Accuracy is:**

$$
\text{Accuracy}
=
\frac{\text{Correct Predictions}}
{\text{Total Predictions}}
$$

---

# 🎯 **Precision**

**Precision answers:**

> **Among the examples predicted as a class, how many were actually that class?**

For a class:

$$
\text{Precision}
=
\frac{TP}{TP+FP}
$$

---

# 🔍 **Recall**

**Recall answers:**

> **Among the examples that truly belong to a class, how many did the model find?**

**For a class:**

$$
\text{Recall}
=
\frac{TP}{TP+FN}
$$

---

# ⚖️ **F1 Score**

**F1 combines precision and recall:**

$$
F1
=
2
\times
\frac{\text{Precision}\times\text{Recall}}
{\text{Precision}+\text{Recall}}
$$

**So the notebook evaluates the classifier using:**

```text
Accuracy
Precision
Recall
F1
```

with precision, recall, and F1 computed using **macro averaging**.

---

# 🏷️ **Rename the Target Column**

**Before training:**

```python
tokenize_data = tokenize_data.rename_column(
    "Product",
    "labels"
)
```

This is an important preparation step.

**Before renaming:**

```text
Product
```

**After renaming:**

```text
labels
```

The model/training workflow can then recognize the column as the target labels.

---

# 👀 **Inspect the Final Tokenized Dataset**

```python
tokenize_data
```

At this stage, the dataset contains the model inputs plus the target labels.

**Conceptually:**

```text
input_ids
attention_mask
token_type_ids
labels
```

---

# 🏋️ **Create the Trainer**

**The notebook imports:**

```python
from transformers import Trainer
```

**Then:**

```python
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenize_data["train"],

    eval_dataset=tokenize_data["test"],

    compute_metrics=compute_metrics
)
```

**The `Trainer` connects the major components:**

```text
Model
+
Training configuration
+
Training dataset
+
Evaluation dataset
+
Metrics
        ↓
      Trainer
```

This avoids manually writing the complete PyTorch training loop.

---

# 🚀 **Fine-Tune the Model**

```python
trainer.train()
```

This is the point where the pretrained BERT model is actually adapted to the complaint-classification task.

**Conceptually:**

```text
Pretrained BERT
      ↓
Complaint training data
      ↓
Forward pass
      ↓
Compute classification loss
      ↓
Backpropagation
      ↓
Update parameters
      ↓
Repeat
```

After training, the model should have learned task-specific patterns connecting complaint narratives to the five target classes.

---

# 💾 **Save the Fine-Tuned Model**

```python
trainer.save_model(
    "/content/drive/MyDrive/NLP and Gen-AI Batches/27 Jun 26 -  Velocity GenAI & Agentic AI Batch/Daily Class Notes/26_08_21_BERT_FineTuning/model"
)
```

This saves the fine-tuned model to the specified directory.

**Conceptually:**

```text
Fine-tuned BERT
      ↓
Save
      ↓
model/
```

Saving the model allows the trained model to be reused later without retraining from scratch.

---

# 🔄 **Reload the Fine-Tuned Model**

**The notebook reloads the saved model:**

```python
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/NLP and Gen-AI Batches/27 Jun 26 -  Velocity GenAI & Agentic AI Batch/Daily Class Notes/26_08_21_BERT_FineTuning/model"
)

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/NLP and Gen-AI Batches/27 Jun 26 -  Velocity GenAI & Agentic AI Batch/Daily Class Notes/26_08_21_BERT_FineTuning/model"
)
```

This demonstrates that the saved artifact can later be loaded for inference.

---

# 🔮 **Make Predictions**

**The notebook uses a Hugging Face pipeline:**

```python
from transformers import pipeline

pipe = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer
)
```

The pipeline hides much of the lower-level inference code.

**Conceptually:**

```text
Text
 ↓
Tokenizer
 ↓
BERT
 ↓
Classification scores
 ↓
Predicted label
```

---

# 📝 **Example Prediction**

```python
result = pipe("my loan is paid ")

result
```

The exact output may contain a model-generated label and score.

**For example, conceptually:**

```text
[
    {
        "label": "LABEL_0",
        "score": ...
    }
]
```

The score represents the model's confidence-like probability output for the selected class.

---

# 🗺️ **Inspect the Model's Label Mapping**

**The notebook runs:**

```python
model.config.label2id
```

**and obtains:**

```text
{
    'LABEL_0': 0,
    'LABEL_1': 1,
    'LABEL_2': 2,
    'LABEL_3': 3,
    'LABEL_4': 4
}
```

This tells us how the model currently identifies its output labels.

**Conceptually:**

```text
LABEL_0 → 0
LABEL_1 → 1
LABEL_2 → 2
LABEL_3 → 3
LABEL_4 → 4
```

---

# 🏷️ **Convert Model Labels into Human-Readable Classes**

**The notebook then defines:**

```python
class_label = {

    'LABEL_0': 'Loan',

    'LABEL_1': 'Card',

    'LABEL_2': 'Services',

    'LABEL_3': 'Credit_report',

    'LABEL_4': 'Others'

}
```

**Then:**

```python
class_label[result[0]['label']]
```

**produces:**

```text
Others
```

---

# ⚠️ **Important: Keep the Class Mapping Consistent**

There is an important detail worth noticing in the notebook.

**Earlier, the numeric mapping was explicitly created as:**

```python
classes = {

    "Loan": 0,

    "Credit_Reporting": 1,

    "Card": 2,

    "Service": 3,

    "Others": 4

}
```

**Therefore the training-time semantic mapping was:**

```text
0 → Loan
1 → Credit_Reporting
2 → Card
3 → Service
4 → Others
```

**Later, the notebook defines:**

```python
class_label = {

    'LABEL_0': 'Loan',

    'LABEL_1': 'Card',

    'LABEL_2': 'Services',

    'LABEL_3': 'Credit_report',

    'LABEL_4': 'Others'
}
```

**That later mapping implies:**

```text
0 → Loan
1 → Card
2 → Services
3 → Credit_report
4 → Others
```

These two mappings are **not identical**.

So the final human-readable interpretation should be checked carefully against the labels used during training.

> **Best practice:** Keep one single authoritative mapping between class IDs and class names throughout dataset preparation, training, saving, and inference.

**For example:**

```python
id2label = {
    0: "Loan",
    1: "Credit_Reporting",
    2: "Card",
    3: "Service",
    4: "Others"
}

label2id = {
    "Loan": 0,
    "Credit_Reporting": 1,
    "Card": 2,
    "Service": 3,
    "Others": 4
}
```

Then use that same mapping consistently.

---

# 🧠 **How the Complete BERT Fine-Tuning Pipeline Works**

```text
             Raw Complaint Dataset
                       ↓
               Select useful columns
                       ↓
                  Drop nulls
                       ↓
              Group original classes
                       ↓
               Clean complaint text
                       ↓
              Check class distribution
                       ↓
                  Undersampling
                       ↓
               Numeric class labels
                       ↓
         Hugging Face Dataset conversion
                       ↓
                Train / Test split
                       ↓
             BERT tokenizer + model
                       ↓
                  Tokenization
                       ↓
              input_ids / masks
                       ↓
             Training configuration
                       ↓
               Evaluation metrics
                       ↓
                  Trainer
                       ↓
                 trainer.train()
                       ↓
                Fine-tuned BERT
                       ↓
                  Save model
                       ↓
                 Reload model
                       ↓
              Text-classification pipeline
                       ↓
                    Prediction
                       ↓
             Human-readable class
```

---

# 🧩 **What BERT Is Learning During Fine-Tuning**

**The model receives:**

```text
Complaint narrative
```

**and tries to predict:**

```text
Loan
Credit_Reporting
Card
Service
Others
```

**For example:**

```text
"my loan is paid but ..."
            ↓
          BERT
            ↓
      Classification Head
            ↓
    [score₀, score₁, score₂, score₃, score₄]
            ↓
         argmax
            ↓
      Predicted Class
```

During training, the model compares the predicted distribution against the true class label and updates its parameters.

---

# 📐 **Classification Loss — Conceptual View**

**For a single example with true class \(y\), the classifier predicts probabilities:**

$$
P(y=0),P(y=1),\ldots,P(y=C-1)
$$

**The usual single-label classification objective is cross-entropy:**

$$
\mathcal{L}
=
-\log P(y\mid x)
$$

**where:**

```text
x → input complaint
y → correct class
```

Across many examples, the training process minimizes the average loss.

This is how the pretrained BERT parameters become specialized for the complaint classification task.

---

# 🔄 **Training vs Fine-Tuning — Side by Side**

| Feature | Training from Scratch | Fine-Tuning |
|---|---|---|
| Initial parameters | Random | Pretrained |
| Language knowledge at start | None | Already learned |
| Task-specific data required | Usually much larger | Often less |
| Compute requirement | High | Lower relative to full pretraining |
| Main goal | Learn everything | Adapt existing knowledge |
| Example | Train BERT from random initialization | Fine-tune `bert-base-cased` |

---

# 📊 **Dataset Preparation vs Model Preparation**

It is helpful to separate the notebook into two phases.

## **Phase 1 — Data Preparation**

```text
Read CSV
↓
Select columns
↓
Drop null values
↓
Map classes
↓
Preprocess text
↓
Check imbalance
↓
Undersample
↓
Convert labels to integers
↓
Create Hugging Face Dataset
↓
Train / Test split
```

## **Phase 2 — Model Preparation + Fine-Tuning**

```text
Load checkpoint
↓
Load tokenizer
↓
Load classification model
↓
Tokenize data
↓
Set training arguments
↓
Define metrics
↓
Rename target to labels
↓
Create Trainer
↓
Train
↓
Save
↓
Reload
↓
Predict
```

---

# 🧠 **Important BERT Concepts in This Notebook**

## **1. Pretrained Checkpoint**

```python
checkpoint = "bert-base-cased"
```

This is the starting point for transfer learning.

---

## **2. Tokenizer**

```python
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
```

Converts text into model-readable inputs.

---

## **3. Sequence Classification Model**

```python
AutoModelForSequenceClassification
```

Adds a classification head on top of the pretrained encoder.

---

## **4. `num_labels=5`**

```python
num_labels=5
```

Tells the classification head that there are five output classes.

---

## **5. Tokenization**

```python
input_ids
attention_mask
token_type_ids
```

These become the model inputs.

---

## **6. Labels**

```python
labels
```

These represent the correct target class for each training example.

---

## **7. Trainer**

```python
Trainer(...)
```

Coordinates training and evaluation.

---

## **8. Metrics**

```text
Accuracy
Precision
Recall
F1
```

These measure classification performance.

---

# ⚠️ **Important Practical Observations**

### **BERT is not automatically better just because it is pretrained**

**Pretraining gives the model a strong starting point, but task performance still depends on:**

```text
Quality of dataset
Correct labels
Class distribution
Preprocessing
Tokenization
Training configuration
Evaluation methodology
```

---

### **Undersampling is not the only way to address class imbalance**

**This notebook uses:**

```text
Undersampling
```

**Other approaches can include:**

```text
Class-weighted loss
Oversampling
Data augmentation
Threshold / decision adjustments
```

The correct choice depends on the dataset and task.

---

### **Aggressive preprocessing should be evaluated**

**The notebook removes:**

```text
Uppercase differences
Masking strings
Non-alphabetic characters
```

This may help in some datasets, but it also removes information such as punctuation and numbers.

> **The right preprocessing strategy should be validated empirically rather than assumed to improve BERT.**

---

### **One epoch is only a configuration choice**

```python
num_train_epochs=1
```

does not mean one epoch is universally sufficient.

It simply means this notebook's training run uses one epoch.

For a real project, training should be evaluated against validation performance and checked for underfitting or overfitting.

---

# 📚 **Quick Revision**

```text
Training
→ Parameters initialized randomly

Fine-Tuning
→ Parameters initialized from a pretrained model
→ Fine-tune on task-specific data

Data Gathering
→ Load the complaints dataset

Column Selection
→ Keep Product + Consumer complaint narrative

Null Handling
→ Remove rows with missing values

Class Mapping
→ Combine many original products into broader classes

Preprocessing
→ Lowercase
→ Remove masking
→ Remove non-alpha characters

Class Imbalance
→ Some classes have more examples than others

Undersampling
→ Select a fixed number of examples from each class

Class IDs
→ Loan = 0
→ Credit_Reporting = 1
→ Card = 2
→ Service = 3
→ Others = 4

Hugging Face Dataset
→ Convert pandas DataFrame using Dataset.from_pandas()

Train/Test Split
→ 80% training
→ 20% test

Tokenizer
→ Converts text into BERT input representations

BERT
→ Pretrained encoder

Sequence Classification Head
→ Converts BERT representations into class scores

num_labels
→ 5

Tokenization
→ input_ids
→ token_type_ids
→ attention_mask

max_length
→ 500 in this notebook

truncation=True
→ Cut sequences longer than the limit

padding="max_length"
→ Pad shorter sequences to the configured length

TrainingArguments
→ Configure training behavior

Metrics
→ Accuracy
→ Macro Precision
→ Macro Recall
→ Macro F1

Trainer
→ High-level Hugging Face training loop

trainer.train()
→ Fine-tune BERT

trainer.save_model()
→ Save fine-tuned model

from_pretrained()
→ Reload saved model

pipeline("text-classification")
→ Perform inference

label2id / class_label
→ Translate model output labels into human-readable classes
```

---

# 🧭 **Final Mental Model**

```text
                PRETRAINED BERT
                      │
                      ↓
          General Language Knowledge
                      │
                      ↓
             Complaint Dataset
                      │
          ┌───────────┴───────────┐
          ↓                       ↓
     Text Cleaning            Label Mapping
          │                       │
          └───────────┬───────────┘
                      ↓
                Undersampling
                      ↓
            Hugging Face Dataset
                      ↓
               Train / Test Split
                      ↓
                 Tokenization
                      ↓
              BERT + Classifier
                      ↓
                Fine-Tuning
                      ↓
               Evaluate Metrics
                      ↓
                 Save Model
                      ↓
               Reload Model
                      ↓
                  New Text
                      ↓
                 Prediction
                      ↓
             Complaint Category
```

---

# 🎯 **Core Takeaways**

> **1. Training from scratch starts with randomly initialized parameters, whereas fine-tuning starts from pretrained parameters.**

> **2. BERT is pretrained to learn general language representations, and fine-tuning adapts those representations to a downstream task.**

> **3. In this notebook, the downstream task is five-class consumer complaint classification.**

> **4. Data preparation is just as important as model selection: null handling, class grouping, preprocessing, balancing, and label encoding all affect training.**

> **5. Undersampling creates a more balanced dataset by reducing the number of examples in larger classes.**

> **6. The Hugging Face tokenizer converts complaint text into `input_ids`, `attention_mask`, and `token_type_ids`.**

> **7. `AutoModelForSequenceClassification` adds a task-specific classification head on top of the pretrained BERT model.**

> **8. The classification head may be newly initialized, which explains the warning about missing parameters from the original checkpoint.**

> **9. `Trainer` simplifies the fine-tuning loop by combining the model, data, training arguments, evaluation dataset, and metrics.**

> **10. Saving and reloading the fine-tuned model makes it possible to use the trained classifier later without retraining.**

> **11. The final prediction should always be interpreted using one consistent class-ID-to-class-name mapping.**

> **12. The overall idea is:**

```text
Pretrained Knowledge
        ↓
Task-Specific Data
        ↓
Fine-Tuning
        ↓
Specialized BERT Model
        ↓
Task Prediction
```
